Import relevant libraries and import the file using a relative path.
Then Split the data into test and training. We use a specific seed as our random state so that we get the same result every time.

## Imports
- **pandas** is a library for data manipulation. We will use the pandas dataframe as it can easily be turned into PyTorch
- **pathlib** a smart little library used to make relative paths. This way we can have a path that is the same for everyone.
- **torch** The Pytorch library.
- **tranformers** we import AutoConfig and AutoModel from the HugginFace tranformer module. These are used to get the configuration from one of HugginFaces models, and then we create a model from that config using the AutoModel function.
- **train_test_split** this is used to split data into training and testing.

In [ ]:
import pandas as pd
from pathlib import Path
import matplotlib.pyplot as plt
import torch 
import torch.nn as nn #neural network module
import torch.nn.functional as F #functional module contains functions that don't have parameters, like activation functions and loss functions
from transformers import AutoConfig, AutoModel # AutoConfig is used to load the configuration of a pre-trained model, and AutoModel is used to load the pre-trained model itself.
from sklearn.model_selection import train_test_split #train_test_split is a function from scikit-learn that splits arrays or matrices into random train and test subsets.
from torch.utils.data import Dataset, DataLoader #Dataset is an abstract class representing a dataset, and DataLoader is a utility that provides an iterable over the given dataset.

,R1-PA1:VH,R1-PM1:V,R1-PA2:VH,R1-PM2:V,R1-PA3:VH,R1-PM3:V,R1-PA4:IH,R1-PM4:I,R1-PA5:IH,R1-PM5:I,...,control_panel_log3,control_panel_log4,relay1_log,relay2_log,relay3_log,relay4_log,snort_log1,snort_log2,snort_log3,snort_log4
2484,-172.781153,139783.4803,67.270975,139783.4803,-52.717847,139883.7733,0.000000,0.00000,0.000000,0.00000,...,0,0,1,0,0,1,0,0,0,0
2750,-74.392840,131634.6675,165.642099,131609.5942,45.641818,131709.8873,-76.105984,384.16478,164.066465,385.08033,...,0,0,0,0,0,0,0,0,0,0
2503,169.721558,132111.0596,49.744196,132085.9864,-70.261814,132186.2794,169.394972,304.32882,49.228534,306.34303,...,0,0,0,0,0,0,0,0,0,0
80,20.637940,132813.1112,-99.327963,132788.0379,140.666868,132888.3310,21.205168,280.70763,-98.823761,279.42586,...,0,0,0,0,0,0,0,0,0,0
803,-74.736615,132261.4993,165.292594,132236.4260,45.286584,132361.7923,-72.656778,267.15749,167.309406,267.15749,...,0,0,0,0,0,0,0,0,0,0


# Prepare Data
Import the dataset using the path variable. Then use the train_test_split() funtion to split the data into random train and test subsets.

bla bla bla... more text is coming.

In [ ]:
"""
# Split the data into training and testing sets and encode the target variable 'marker' as 0 for 'Natural' and 1 for 'Attack'
X_train, X_test, y_train, y_test = train_test_split(df.drop('marker', axis=1), df['marker'].map({'Natural': 0, 'Attack': 1}), test_size=0.2, random_state=42)
"""

path = Path('datasets/binaryAllNaturalPlusNormalVsAttacks/data1.csv')
df = pd.read_csv(path)
IN_CHANNELS = df.shape[1] - 1 #the number of features is the number of columns minus one (the target variable 'marker')
#print(f'Number of features: {IN_CHANNELS}')
NUM_CLASSES = 2 #the number of classes is 2 (Natural and Attack)

X = df.drop('marker', axis=1) #drop the target variable 'marker' from the dataframe to create the feature matrix X
y = df['marker'].map({'Natural': 0, 'Attack': 1}) #map the target variable 'marker' to 0 for 'Natural' and 1 for 'Attack' to create the target variable y

class TimeSeriesDataset(Dataset):
    def __init__(self, X, y, window_size=32):
        # Convert pandas data into PyTorch tensors
        self.X = torch.tensor(X.values, dtype=torch.float32)
        self.y = torch.tensor(y.values, dtype=torch.float32)

        # Save the number of timesteps per window
        self.window_size = window_size

    def __len__(self):
        # Number of possible sliding windows
        return len(self.X) - self.window_size + 1

    def __getitem__(self, idx):
        # Take consecutive rows from idx up to idx + window_size
        window = self.X[idx : idx + self.window_size]

        # Use the label of the last row in the window
        # Might be better to label according to wether any of the rows in the window are an attack
        label = self.y[idx + self.window_size - 1]

        return window, label

# Split the data into training and testing sets using an 80-20 split
# Note, need to check if there is a more clean way to do this
split_idx = int(0.8 * len(X))

X_train = X.iloc[:split_idx]
X_test = X.iloc[split_idx:]

y_train = y.iloc[:split_idx]
y_test = y.iloc[split_idx:]

# Plotting features

This is how to plot features using matplotlib.
Might become valuable if we want to analyse specific features later.

In [ ]:
#Hvordan man plotter en feature? (Måske er det nydvendigt senere)
df_numeric = df.apply(pd.to_numeric, errors="coerce")

# Plot a specific feature, for example 'R1-PA1:VH'
plt.plot(df_numeric[['R1-PA1:VH']])
plt.xlabel("Sample index")
plt.ylabel("Feature value")
plt.title("Specific Feature over samples")
plt.show()

# Plot the first 5 features as an example
plt.plot(df_numeric.iloc[:, 0:5])
plt.xlabel("Sample index")
plt.ylabel("Feature value")
plt.title("First 5 Features over samples")
plt.show()


# Definition af CNN feature extractor

- self.cnn = nn.Sequential()
    - order of inputs define which order each layer should execute in

- nn.Conv1d(in_channels, 32, kernel_size=3, stride=2, padding=1)
    - in_channels : Number of input features, so how many variables are in each timestep
    - 32 : Output channels. How many patterns the CNN will learn.
    - Kernel_size : Size of the filter
    - Stride : how many steps we move the filter
    - padding : how many 0 we add to the start and end of the input. Prevents the sequence from shrinking too much. 

- nn.BatchNorm1d(32)
    - Normalizes the activations 
    
- nn.ReLU(inplace=True)
    - Means we are using relu
    - Inplace : modifys the current tensor directly, instead of creating a whole new tensor. This saves memory. 

- nn.MaxPool1d(2)
    - Reduce the sequence lenght by only keeping only the strongest activations
    - the 2, means that the seqence lenght is halved

In [ ]:
class FeatureExtractor(nn.Module):
    def __init__(self, in_channels):
        super().__init__()

        self.features = nn.Sequential(
            nn.Conv1d(in_channels, 64, kernel_size=5, stride=1, padding=2),
            nn.BatchNorm1d(64),
            nn.ReLU(),
            nn.MaxPool1d(2), # reduce the sequence length by half

            nn.Conv1d(64, 128, kernel_size=5, stride=1, padding=2),
            nn.BatchNorm1d(128),
            nn.ReLU(),
            nn.MaxPool1d(2), 

            nn.Conv1d(128, 256, kernel_size=3, stride=1, padding=1),
            nn.BatchNorm1d(256),
            nn.ReLU()
        )
        
    def forward(self, x):
        return self.features(x)

# Classifier
This module implements the final classification head of the model. Its purpose is to thransform the feature representation produced by the FeatureExtractor into class logits.

The classifier takes a feature vectore and predicts the probability of each class.

- Flatten
    - Converts the input tensor into a 1-dimentional feature vector
- Linear Layer
    - Fully connected layer that learns combinations of the extracted features.
    - Reduces the feature dimension
- ReLU
    - Applies the non-linear function `f(x) = max(0, x)`
- Dropout
    - Randomly disables 20% of neurons during training.
    - Helps prevent overfitting by making the model rely on multiple features instead of a few dominant ones
- Linear Layer
    - Produces the final logits for each class.
    - For binary classification (`num_classes = 2`), the output shape becomes: `(batch_size, 2)`

In [ ]:
class Classifier(nn.Module):
    def __init__(self, d_model):
        super().__init__()
        self.classifier = nn.Sequential(
            nn.Linear(d_model, 128),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(128, 1) # single output for binary classification
        )

    def forward(self, x):
        return self.classifier(x)

# Transformer-Encoder

# Definition af CNNTRansformer klasse 

- num_classes : Number of classes we want to classify prediction in. In our case it is binary (True / False)

- in_channels : Number of columns in dataset, or number of culumns the model should analyze. In our case it is all columns, expect for timestep. According to chat, we don't need timestep to make preidiction. 

- embed_dim : size of feature vector used by transformer. if training is unstable, reduce to 64, if model overfits, use 256. Embed_dim must be divisable by num_heads (FInd ud af specifikt h)

- num_heads : Number of attention heads in transformers mult-head attention layer. 

- num_layers : Number of transformer layers (?)

- mlp_dim : Size of feedforward inside each transformor layer (Hvorfor er det 256)

- dropout : Percentage of neurons we randomly turn off during each training step. 

In [ ]:
class CNNTransformerBinary(nn.Module):
    def __init__(self, in_channels, d_model=256, nhead=4, num_layers=2):
        super().__init__()
        loss = nn.BCEWithLogitsLoss() # Binary Cross Entropy Loss with logits, suitable for binary classification tasks

        # CNN feature extractor
        self.feature_extractor = FeatureExtractor(in_channels)

        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model, 
            nhead=nhead, 
            batch_first=True
        )

        self.transformer = nn.TransformerEncoder(encoder_layer, num_layers=num_layers)

        self.classifier = Classifier(d_model)


    def forward(self, x):
        # x: (B, C, L)
        x = self.feature_extractor(x)   # (B, 256, L')
        x = x.transpose(1, 2)           # (B, L', 256)
        x = self.transformer(x)         # (B, L', 256)
        x = x.mean(dim=1)               # (B, 256)
        x = self.classifier(x)          # (B, 1)
        return x

In [62]:
model = CNNTransformer()
print(model)

CNNTransformer(
  (cnn): FeatureExtractor(
    (features): Sequential(
      (0): Conv1d(128, 64, kernel_size=(5,), stride=(1,), padding=(2,))
      (1): BatchNorm1d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (2): ReLU()
      (3): MaxPool1d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
      (4): Conv1d(64, 128, kernel_size=(5,), stride=(1,), padding=(2,))
      (5): BatchNorm1d(128, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (6): ReLU()
      (7): MaxPool1d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
      (8): Conv1d(128, 256, kernel_size=(3,), stride=(1,), padding=(1,))
      (9): BatchNorm1d(256, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (10): ReLU()
      (11): AdaptiveAvgPool1d(output_size=1)
    )
  )
  (classifier): Linear(in_features=256, out_features=2, bias=True)
)


I test eksempelet ovenover kører vi med denne rækkefølge : 

x -> CNN encoder -> z -> Hugging Face BERT -> z' -> decoder -> x_hat

Hvor : 
- z = encoded features
- z' = transformer processed features 
- x_hat = reconstructed signal 

Så : 
- Input kommer med x, som er ...
- Encoder virker som vores feature extractor 
- Transformer lærer relationer mellem time series i de extracted features 
- Decoder prøver at rekonstruerer signalet? 
- x_hat er så det rekonstruerede output 

# Transformer configuration 
- Config : Gets the  configuration of the pretrained model, in our case bert, which is from huggingface
- Model : generates the model from the configuration, initialized with randomized weights

The config, defines the following:
- hidden size : size of the vector used to represent each token inside the model. So the size of how many numbers should represent a token/word, and that becomes a vector. Must be divisible by number of heads. 

- number of layers : Number of transformer blocks stacked on top of eachother. Contains mult head layer connections. 12 layers. 

- number of attention heads : The amount of attention heads running in parralel. So each head would focus on different attention pattern (Long term relation, grammar, closely-related, etc.)

- intermediate size : hidden size of the feedforward network inside each tranformer network. Each layer has a small neural network. 
    - linear layer -> Activation function -> Linear layer
    - Often size = 4 * hidden size?

- dropout values : Regularization technique. Randomly turns of a percentage of random neurons. 

- vocabulary size : How many unqiue tokens a language knows. 

- positional embedding length : Defines maximum sequence length. So a model can only process at most x amount of tokens at once. Each position has a vector. For time series models this corresponds to max window size (sequence length).
    - final embedding = token embedding + positional embedding.


